In [ ]:
# =============================================================================
# CUSTOMIZABLE DATASET INDEXING & FUSION SYSTEM WITH VISUALIZATION
# Supports SID-SET, DF2023, SO-Fake-set, and custom datasets
# =============================================================================

import os
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Callable, Any
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import random
import gc
import warnings
warnings.filterwarnings("ignore")

# Visualization imports
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import cv2

# =============================================================================
# REPRODUCIBILITY
# =============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =============================================================================
# CONFIGURATION SYSTEM
# =============================================================================

@dataclass
class DatasetConfig:
    """Configuration for a single dataset."""
    name: str
    path: Path
    label: int  # 0 = real, 1 = fake
    category: str  # "REAL", "TAMPERED", "SYNTHETIC", "LOCALIZATION", "SO_FAKE"
    image_extension: str = "*"
    mask_path: Optional[Path] = None
    mask_extension: str = "*"
    max_samples: Optional[int] = None
    recursive: bool = True
    label_mapping: Dict[str, int] = field(default_factory=dict)
    custom_loader: Optional[Callable] = None
    description: str = ""  # Optional description
    
@dataclass
class DatasetRegistry:
    """Registry for managing multiple datasets."""
    datasets: List[DatasetConfig] = field(default_factory=list)
    global_max_samples: Optional[int] = None
    
    def add_dataset(self, config: DatasetConfig) -> None:
        self.datasets.append(config)
    
    def get_dataset_names(self) -> List[str]:
        return [d.name for d in self.datasets]

# =============================================================================
# CUSTOM DATA RECORD
# =============================================================================

@dataclass
class ForensicRecord:
    """Flexible forensic data record."""
    image_path: str
    mask_path: str = ""
    label: int = 0
    source: str = ""
    category: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

# =============================================================================
# IMAGE COLLECTION UTILITIES
# =============================================================================

def collect_images(
    folder: Path,
    extensions: set = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"},
    recursive: bool = True,
    max_files: Optional[int] = None
) -> List[Path]:
    """Collect image files from a folder."""
    if not folder.exists():
        return []
    
    files = []
    for ext in extensions:
        if recursive:
            files.extend(folder.rglob(f"*{ext}"))
        else:
            files.extend(folder.glob(f"*{ext}"))
    
    files = sorted(files)
    if max_files:
        files = files[:max_files]
    return files

def find_matching_masks(
    image_path: Path,
    mask_folder: Path,
    extensions: set = {".png", ".jpg", ".jpeg", ".tif", ".bmp"}
) -> List[Path]:
    """Find matching mask files for a given image."""
    if not mask_folder.exists():
        return []
    
    stem = image_path.stem
    matches = []
    for ext in extensions:
        matches.extend(mask_folder.rglob(f"{stem}*{ext}"))
    return matches

# =============================================================================
# DATASET INDEXER
# =============================================================================

class DatasetIndexer:
    """
    Flexible dataset indexer that can handle multiple dataset formats.
    Supports custom loading functions and label mappings.
    """
    
    def __init__(self, registry: DatasetRegistry):
        self.registry = registry
        self.records = []
        self._image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
        
    def index_datasets(self) -> pd.DataFrame:
        """Index all datasets in the registry and return a unified DataFrame."""
        self.records = []
        
        for config in self.registry.datasets:
            if config.custom_loader:
                # Use custom loader if provided
                records = config.custom_loader(config)
                self.records.extend(records)
            else:
                # Use default loader
                records = self._index_single_dataset(config)
                self.records.extend(records)
        
        return self.to_dataframe()
    
    def _index_single_dataset(self, config: DatasetConfig) -> List[ForensicRecord]:
        """Index a single dataset with default logic."""
        records = []
        
        # Handle label mapping
        label = config.label
        
        # Collect images
        images = collect_images(
            config.path,
            extensions=self._image_extensions,
            recursive=config.recursive,
            max_files=config.max_samples
        )
        
        for img_path in tqdm(images, desc=f"Indexing {config.name}"):
            mask_path = ""
            
            # Find matching mask if mask path is provided
            if config.mask_path:
                matches = find_matching_masks(
                    img_path,
                    config.mask_path,
                    extensions={".png", ".jpg", ".jpeg", ".tif", ".bmp"}
                )
                if matches:
                    mask_path = str(matches[0])
            
            record = ForensicRecord(
                image_path=str(img_path),
                mask_path=mask_path,
                label=label,
                source=config.name,
                category=config.category,
                metadata={
                    "original_name": img_path.name,
                    "stem": img_path.stem,
                    "description": config.description,
                }
            )
            records.append(record)
        
        return records
    
    def to_dataframe(self) -> pd.DataFrame:
        """Convert records to a unified DataFrame."""
        return pd.DataFrame([vars(r) for r in self.records])
    
    def shuffle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Shuffle the dataset."""
        return df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# =============================================================================
# DATASET SPLITTER
# =============================================================================

class DatasetSplitter:
    """Flexible dataset splitter with stratification support."""
    
    def __init__(self, 
                 test_size: float = 0.20,
                 val_size: float = 0.50,
                 random_state: int = SEED):
        self.test_size = test_size
        self.val_size = val_size
        self.random_state = random_state
    
    def split(self, df: pd.DataFrame, stratify_col: str = "label") -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Split dataset into train, validation, and test sets."""
        
        # First split: train and temp
        train_df, temp_df = train_test_split(
            df,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=df[stratify_col]
        )
        
        # Second split: validation and test
        val_df, test_df = train_test_split(
            temp_df,
            test_size=self.val_size,
            random_state=self.random_state,
            stratify=temp_df[stratify_col]
        )
        
        return (train_df.reset_index(drop=True),
                val_df.reset_index(drop=True),
                test_df.reset_index(drop=True))

# =============================================================================
# VISUALIZATION UTILITIES
# =============================================================================

def load_image(path: str, size: Tuple[int, int] = (224, 224)) -> np.ndarray:
    """Load and resize an image."""
    try:
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((size[0], size[1], 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, size)
        return img
    except:
        return np.zeros((size[0], size[1], 3), dtype=np.uint8)

def display_image_grid(
    images: List[np.ndarray],
    titles: List[str] = None,
    cols: int = 4,
    figsize: Tuple[int, int] = (16, 10)
) -> None:
    """Display a grid of images with optional titles."""
    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(images[i])
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=10)
            ax.axis('off')
        else:
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

def visualize_dataset_samples(
    df: pd.DataFrame,
    dataset_name: str,
    n_samples: int = 6,
    cols: int = 3,
    show_masks: bool = True,
    figsize: Tuple[int, int] = (15, 10)
) -> None:
    """
    Display sample images (and masks) from a dataset.
    
    Args:
        df: DataFrame with 'image_path' and optionally 'mask_path'
        dataset_name: Name of dataset for title
        n_samples: Number of samples to display
        cols: Number of columns
        show_masks: Whether to show masks alongside images
        figsize: Figure size
    """
    # Filter by source
    df_subset = df[df['source'] == dataset_name]
    if len(df_subset) == 0:
        print(f"No samples found for dataset: {dataset_name}")
        return
    
    # Random sample
    sample_df = df_subset.sample(min(n_samples, len(df_subset)), random_state=SEED)
    
    # Prepare images and masks
    images = []
    masks = []
    titles = []
    
    for idx, row in sample_df.iterrows():
        img = load_image(row['image_path'])
        images.append(img)
        
        # Determine label text
        label_text = "Real" if row['label'] == 0 else "Fake"
        titles.append(f"{label_text} ({row['category']})")
        
        # Mask
        if show_masks and row['mask_path'] and os.path.exists(row['mask_path']):
            mask = load_image(row['mask_path'], size=(224, 224))
            # Convert to grayscale for mask display
            if mask.ndim == 3:
                mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
            masks.append(mask)
        else:
            masks.append(None)
    
    # Display images
    print(f"\n{dataset_name} - Sample Images:")
    display_image_grid(images, titles, cols=cols, figsize=figsize)
    
    # Display masks if any
    if show_masks and any(m is not None for m in masks):
        mask_titles = [f"Mask {i+1}" for i in range(len(masks))]
        # Replace None with blank image
        mask_images = []
        for m in masks:
            if m is not None:
                mask_images.append(m)
            else:
                mask_images.append(np.zeros((224, 224), dtype=np.uint8))
        print(f"\n{dataset_name} - Corresponding Masks:")
        display_image_grid(mask_images, mask_titles, cols=cols, figsize=figsize)

def visualize_dataset_distribution(df: pd.DataFrame, figsize: Tuple[int, int] = (14, 6)):
    """Display bar charts for dataset distribution by source and category."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Source distribution
    source_counts = df['source'].value_counts()
    axes[0].bar(source_counts.index, source_counts.values, color='skyblue')
    axes[0].set_title('Samples per Dataset', fontsize=14)
    axes[0].set_xlabel('Dataset')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Category distribution
    cat_counts = df['category'].value_counts()
    axes[1].bar(cat_counts.index, cat_counts.values, color='lightcoral')
    axes[1].set_title('Samples per Category', fontsize=14)
    axes[1].set_xlabel('Category')
    axes[1].set_ylabel('Count')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

# =============================================================================
# DATASET CONFIGURATION BUILDER
# =============================================================================

class DatasetBuilder:
    """Builder pattern for creating dataset configurations with specific paths."""
    
    @staticmethod
    def create_sid_set(
        root_path: Path,
        real_subdir: str = "real",
        tampered_subdir: str = "tampered",
        synthetic_subdir: str = "full_synthetic",
        masks_subdir: str = "masks",
        max_samples: Optional[int] = None
    ) -> List[DatasetConfig]:
        """Create SID-SET configurations with given root path."""
        
        root = Path(root_path)
        
        configs = [
            DatasetConfig(
                name="SID_REAL",
                path=root / real_subdir,
                label=0,
                category="REAL",
                description="Real images from SID-SET",
                max_samples=max_samples
            ),
            DatasetConfig(
                name="SID_TAMPERED",
                path=root / tampered_subdir,
                label=1,
                category="TAMPERED",
                mask_path=root / masks_subdir,
                description="Tampered images with masks from SID-SET",
                max_samples=max_samples
            ),
            DatasetConfig(
                name="SID_SYNTHETIC",
                path=root / synthetic_subdir,
                label=1,
                category="SYNTHETIC",
                description="Synthetic images from SID-SET",
                max_samples=max_samples
            )
        ]
        return configs
    
    @staticmethod
    def create_df2023(
        root_path: Path,
        images_subdir: str = "COCO_V15",
        masks_subdir: str = "COCO_V15_GT",
        max_samples: Optional[int] = None,
        dataset_name: str = "DF2023"
    ) -> List[DatasetConfig]:
        """Create DF2023 configuration with given root path."""
        
        root = Path(root_path)
        
        configs = [
            DatasetConfig(
                name=dataset_name,
                path=root / images_subdir,
                label=1,
                category="LOCALIZATION",
                mask_path=root / masks_subdir,
                description="DF2023 dataset with localization masks",
                max_samples=max_samples
            )
        ]
        return configs
    
    @staticmethod
    def create_so_fake_set(
        root_path: Path,
        train_subdir: str = "train",
        val_subdir: str = "validation",
        ood_subdir: str = "ood_test",
        max_samples: Optional[int] = None
    ) -> List[DatasetConfig]:
        """Create SO-Fake-set configurations."""
        
        root = Path(root_path)
        
        configs = [
            DatasetConfig(
                name="SO_FAKE_TRAIN",
                path=root / train_subdir,
                label=1,
                category="SO_FAKE",
                description="SO-Fake-set training split",
                max_samples=max_samples
            ),
            DatasetConfig(
                name="SO_FAKE_VAL",
                path=root / val_subdir,
                label=1,
                category="SO_FAKE",
                description="SO-Fake-set validation split",
                max_samples=max_samples
            ),
            DatasetConfig(
                name="SO_FAKE_OOD",
                path=root / ood_subdir,
                label=1,
                category="SO_FAKE_OOD",
                description="SO-Fake-set out-of-distribution test",
                max_samples=max_samples
            )
        ]
        return configs
    
    @staticmethod
    def create_custom_dataset(
        name: str,
        image_dir: str,
        label: int,
        category: str,
        mask_dir: Optional[str] = None,
        max_samples: Optional[int] = None,
        description: str = "",
        **kwargs
    ) -> DatasetConfig:
        """Create a custom dataset configuration."""
        
        return DatasetConfig(
            name=name,
            path=Path(image_dir),
            label=label,
            category=category,
            mask_path=Path(mask_dir) if mask_dir else None,
            max_samples=max_samples,
            description=description,
            **kwargs
        )

# =============================================================================
# DATASET LOADER - MAIN ENTRY POINT WITH VISUALIZATION
# =============================================================================

class DataLoader:
    """
    Main data loader class for forensic datasets with visualization capabilities.
    """
    
    def __init__(self, 
                 configs: List[DatasetConfig],
                 test_size: float = 0.20,
                 val_size: float = 0.50,
                 output_dir: Optional[Path] = None,
                 save_csv: bool = True,
                 visualize: bool = True):
        
        self.configs = configs
        self.test_size = test_size
        self.val_size = val_size
        self.output_dir = output_dir
        self.save_csv = save_csv
        self.visualize = visualize
        
        # Initialize components
        self.registry = DatasetRegistry()
        for config in configs:
            self.registry.add_dataset(config)
        
        self.indexer = DatasetIndexer(self.registry)
        self.splitter = DatasetSplitter(test_size, val_size)
        
        # Results
        self.df = None
        self.train_df = None
        self.val_df = None
        self.test_df = None
    
    def load(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Load and split the dataset."""
        
        # Index datasets
        print("\n" + "="*70)
        print("INDEXING DATASETS")
        print("="*70)
        self.df = self.indexer.index_datasets()
        self.df = self.indexer.shuffle(self.df)
        
        # Print statistics
        self._print_statistics()
        
        # Visualizations
        if self.visualize:
            self._visualize_overview()
        
        # Split dataset
        self.train_df, self.val_df, self.test_df = self.splitter.split(self.df)
        
        # Save CSVs if requested
        if self.save_csv and self.output_dir:
            self._save_csvs()
        
        return self.train_df, self.val_df, self.test_df
    
    def _print_statistics(self) -> None:
        """Print dataset statistics."""
        print("\n" + "="*70)
        print("DATASET DISTRIBUTION")
        print("="*70)
        
        print("\nLabel Distribution:")
        print(self.df["label"].value_counts())
        
        print("\nBy Source:")
        print(self.df["source"].value_counts())
        
        print("\nBy Category:")
        print(self.df["category"].value_counts())
        
        print(f"\nTotal Samples: {len(self.df):,}")
    
    def _save_csvs(self) -> None:
        """Save dataframes to CSV files."""
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.train_df.to_csv(self.output_dir / "train_df.csv", index=False)
        self.val_df.to_csv(self.output_dir / "val_df.csv", index=False)
        self.test_df.to_csv(self.output_dir / "test_df.csv", index=False)
        
        print(f"\nCSV files saved to {self.output_dir}")
    
    def _visualize_overview(self) -> None:
        """Display visual overview of the dataset."""
        print("\n" + "="*70)
        print("DATASET VISUALIZATION")
        print("="*70)
        
        # Distribution charts
        visualize_dataset_distribution(self.df)
        
        # Sample images from each dataset
        for source in self.df['source'].unique():
            visualize_dataset_samples(
                self.df,
                dataset_name=source,
                n_samples=6,
                cols=3,
                show_masks=True
            )
    
    def summary_table(self) -> pd.DataFrame:
        """Generate summary table for thesis."""
        return pd.DataFrame({
            "Subset": ["Training", "Validation", "Testing", "Total"],
            "Samples": [
                len(self.train_df),
                len(self.val_df),
                len(self.test_df),
                len(self.df)
            ]
        })
    
    def sanity_check(self, n: int = 5) -> None:
        """Perform sanity check on loaded data."""
        print("\n" + "="*70)
        print("SANITY CHECK")
        print("="*70)
        
        print("\nSample Records:")
        display(self.train_df.head(n))
        display(self.val_df.head(n))
        display(self.test_df.head(n))
        
        print(f"\n✓ Data loaded successfully!")

# =============================================================================
# QUICK CONFIGURATION FUNCTIONS WITH PROVIDED PATHS
# =============================================================================

def load_all_datasets(
    sid_root: Path,
    df2023_train_root: Path,
    df2023_val_root: Path,
    so_fake_root: Path,
    sid_max: int = 4000,
    df_max: int = 20000,
    so_fake_max: int = 5000,
    test_size: float = 0.20,
    val_size: float = 0.50,
    output_dir: Optional[Path] = None,
    visualize: bool = True
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Load all available datasets with provided paths.
    
    Args:
        sid_root: SID-SET root path
        df2023_train_root: DF2023 training set root
        df2023_val_root: DF2023 validation set root
        so_fake_root: SO-Fake-set root
        sid_max: Max samples from SID-SET per subfolder
        df_max: Max samples from DF2023 per subfolder
        so_fake_max: Max samples from SO-Fake-set per subfolder
        test_size: Test set proportion
        val_size: Validation set proportion of temp set
        output_dir: Directory to save CSV files
        visualize: Whether to display visual samples
    
    Returns:
        Tuple of (train_df, val_df, test_df)
    """
    
    configs = []
    
    # 1. SID-SET
    configs.extend(DatasetBuilder.create_sid_set(
        sid_root,
        max_samples=sid_max
    ))
    
    # 2. DF2023 Training Set
    configs.extend(DatasetBuilder.create_df2023(
        df2023_train_root,
        dataset_name="DF2023_TRAIN",
        max_samples=df_max
    ))
    
    # 3. DF2023 Validation Set
    configs.extend(DatasetBuilder.create_df2023(
        df2023_val_root,
        dataset_name="DF2023_VAL",
        max_samples=df_max
    ))
    
    # 4. SO-Fake-set
    configs.extend(DatasetBuilder.create_so_fake_set(
        so_fake_root,
        max_samples=so_fake_max
    ))
    
    # Create loader
    loader = DataLoader(
        configs=configs,
        test_size=test_size,
        val_size=val_size,
        output_dir=output_dir,
        visualize=visualize
    )
    
    return loader.load()

# =============================================================================
# MAIN EXECUTION - USING PROVIDED PATHS
# =============================================================================

if __name__ == "__main__":
    
    # Define paths from the user input
    SID_ROOT = Path("/kaggle/input/datasets/mubarekahmed/sida-forensics")
    DF2023_TRAIN_ROOT = Path("/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train")
    DF2023_VAL_ROOT = Path("/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val")
    SO_FAKE_ROOT = Path("/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined")
    OUTPUT_DIR = Path("/kaggle/working")
    
    # Optional: Set sample limits (adjust as needed)
    SID_MAX = 4000          # per subfolder (real, tampered, synthetic)
    DF_MAX = 20000          # per DF2023 set (train/val)
    SO_FAKE_MAX = 5000      # per SO-Fake split (train/val/ood)
    
    # Load all datasets with visualizations
    print("\n" + "="*80)
    print("LOADING ALL DATASETS WITH VISUALIZATION")
    print("="*80)
    
    train_df, val_df, test_df = load_all_datasets(
        sid_root=SID_ROOT,
        df2023_train_root=DF2023_TRAIN_ROOT,
        df2023_val_root=DF2023_VAL_ROOT,
        so_fake_root=SO_FAKE_ROOT,
        sid_max=SID_MAX,
        df_max=DF_MAX,
        so_fake_max=SO_FAKE_MAX,
        output_dir=OUTPUT_DIR,
        visualize=True  # Enable visual samples
    )
    
    print(f"\n✓ Successfully loaded:")
    print(f"  - Training: {len(train_df):,}")
    print(f"  - Validation: {len(val_df):,}")
    print(f"  - Test: {len(test_df):,}")
    print(f"  - Total: {len(train_df) + len(val_df) + len(test_df):,}")

# =============================================================================
# ADDITIONAL CUSTOMIZATION - ADD CUSTOM DATASETS
# =============================================================================

def add_custom_dataset(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    custom_config: Dict[str, Any]
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Add a custom dataset to existing splits.
    
    Args:
        train_df: Existing training DataFrame
        val_df: Existing validation DataFrame
        test_df: Existing test DataFrame
        custom_config: Configuration for custom dataset
    
    Returns:
        Updated DataFrames
    """
    
    # Create config
    config = DatasetBuilder.create_custom_dataset(
        name=custom_config["name"],
        image_dir=custom_config["image_dir"],
        label=custom_config["label"],
        category=custom_config["category"],
        mask_dir=custom_config.get("mask_dir"),
        max_samples=custom_config.get("max_samples"),
        description=custom_config.get("description", "")
    )
    
    # Index single dataset
    registry = DatasetRegistry()
    registry.add_dataset(config)
    indexer = DatasetIndexer(registry)
    df_new = indexer.index_datasets()
    
    # Visualize new dataset if requested
    print(f"\nNew dataset: {custom_config['name']}")
    visualize_dataset_samples(df_new, custom_config['name'], n_samples=6)
    
    # Split the new data (same proportions as before)
    splitter = DatasetSplitter(test_size=0.2, val_size=0.5)
    train_new, val_new, test_new = splitter.split(df_new)
    
    # Concatenate
    train_combined = pd.concat([train_df, train_new], ignore_index=True)
    val_combined = pd.concat([val_df, val_new], ignore_index=True)
    test_combined = pd.concat([test_df, test_new], ignore_index=True)
    
    # Shuffle
    train_combined = train_combined.sample(frac=1, random_state=SEED).reset_index(drop=True)
    val_combined = val_combined.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_combined = test_combined.sample(frac=1, random_state=SEED).reset_index(drop=True)
    
    return train_combined, val_combined, test_combined

# Example usage:
# custom_config = {
#     "name": "MY_CUSTOM",
#     "image_dir": "/path/to/images",
#     "label": 1,
#     "category": "CUSTOM_FAKE",
#     "mask_dir": "/path/to/masks",
#     "max_samples": 1000,
#     "description": "My custom dataset"
# }
# train_df, val_df, test_df = add_custom_dataset(train_df, val_df, test_df, custom_config)